### This notebook contains code that has been produced for comparing the data in the Internet Archive and in the KB (the National Library of the Netherlands) web collections

The data from the KB web collection are under the copyright. Therefore the notebooks contain mentioning the files that were derived and processed. They can be accessed in the KB DataLab.

In [ ]:
# Comparison. Initial Metrics. Overlapping urls

import pandas as pd
import os

# === File paths ===
file_a = r'data/IA_Rijksmuseum_unique_urls_per_year.csv'
file_b = r'data/KB_URLs_200statuscode_full_cleaned.csv'
output_dir = r'data/Results/'
os.makedirs(output_dir, exist_ok=True)

# === Load datasets ===
df_a = pd.read_csv(file_a)
df_b = pd.read_csv(file_b)

# === Rename columns and extract year ===
df_a = df_a.rename(columns={'original': 'url'})
df_b = df_b.rename(columns={'url': 'url'})  # for consistency

df_a['year'] = df_a['timestamp'].astype(str).str[:4]
df_b['year'] = df_b['crawl_year'].astype(str)

# === Drop duplicates and NaNs ===
df_a = df_a.dropna(subset=['url']).drop_duplicates(subset=['url'])
df_b = df_b.dropna(subset=['url']).drop_duplicates(subset=['url'])

# === Convert to sets for comparison ===
urls_a = set(df_a['url'])
urls_b = set(df_b['url'])

# === Intersections and differences ===
intersection = urls_a & urls_b
only_a = urls_a - urls_b
only_b = urls_b - urls_a

# === Mark presence in other file ===
df_a['in_file_b'] = df_a['url'].isin(urls_b)
df_b['in_file_a'] = df_b['url'].isin(urls_a)

# === Yearly overlap distribution ===
overlap_a = df_a[df_a['in_file_b']].groupby('year').size().rename('overlap')
only_a_by_year = df_a[~df_a['in_file_b']].groupby('year').size().rename('only_a')

overlap_b = df_b[df_b['in_file_a']].groupby('year').size().rename('overlap')
only_b_by_year = df_b[~df_b['in_file_a']].groupby('year').size().rename('only_b')

compare_a = pd.concat([overlap_a, only_a_by_year], axis=1).fillna(0).astype(int).sort_index()
compare_b = pd.concat([overlap_b, only_b_by_year], axis=1).fillna(0).astype(int).sort_index()

# === Save CSV outputs ===
compare_a.to_csv(os.path.join(output_dir, 'yearly_comparison_file_a.csv'))
compare_b.to_csv(os.path.join(output_dir, 'yearly_comparison_file_b.csv'))

# === Save summary ===
summary = pd.DataFrame({
    'Metric': ['Total in File A', 'Total in File B', 'Shared URLs', 'Only in A', 'Only in B'],
    'Count': [len(urls_a), len(urls_b), len(intersection), len(only_a), len(only_b)]
})
summary.to_csv(os.path.join(output_dir, 'url_overlap_summary.csv'), index=False)

print("✅ Summary saved to:", output_dir)


In [ ]:
# Overlapping urls with spheric diagram

import pandas as pd
import re
import os
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from urllib.parse import urlparse

# === File Paths ===
file1_path = r'data/IA_Rijksmuseum_unique_urls_per_year.csv'
file2_path = r'data/KB_URLs_200statuscode_full_cleaned.csv'
output_dir = r'data/Results/'

# === Load data ===
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# === Normalize column names ===
if 'original' in df1.columns:
    df1['url'] = df1['original']
if 'original' in df2.columns:
    df2['url'] = df2['original']

# === Extract year ===
def extract_year_from_timestamp(ts):
    match = re.match(r'(\d{4})', str(ts))
    return match.group(1) if match else 'Unknown'

if 'timestamp' in df1.columns:
    df1['year'] = df1['timestamp'].apply(extract_year_from_timestamp)
if 'crawl_year' in df2.columns:
    df2['year'] = df2['crawl_year'].astype(str)

# === Drop rows with missing URLs ===
df1 = df1.dropna(subset=['url'])
df2 = df2.dropna(subset=['url'])

# === Strip trailing slashes for uniformity ===
df1['url'] = df1['url'].str.rstrip('/')
df2['url'] = df2['url'].str.rstrip('/')

# === Create sets for comparison ===
set1 = set(df1['url'].unique())
set2 = set(df2['url'].unique())

# === Overlap and Differences ===
intersection = set1 & set2
only_in_1 = set1 - set2
only_in_2 = set2 - set1

# === Save stats as CSV ===
stats_df = pd.DataFrame({
    'Category': ['Total in IA', 'Total in KB', 'Common URLs', 'Only in IA', 'Only in KB'],
    'Count': [len(set1), len(set2), len(intersection), len(only_in_1), len(only_in_2)]
})
stats_csv_path = os.path.join(output_dir, 'url_overlap_statistics.csv')
stats_df.to_csv(stats_csv_path, index=False)

# === Save URLs by category ===
pd.DataFrame({'url': list(intersection)}).to_csv(os.path.join(output_dir, 'urls_in_both.csv'), index=False)
pd.DataFrame({'url': list(only_in_1)}).to_csv(os.path.join(output_dir, 'urls_only_in_file1.csv'), index=False)
pd.DataFrame({'url': list(only_in_2)}).to_csv(os.path.join(output_dir, 'urls_only_in_file2.csv'), index=False)

# === Print summary ===
print("🔢 Comparison Summary:")
print(stats_df.to_string(index=False))
print(f"\n✅ Statistics saved to: {stats_csv_path}")

# === Plot Venn diagram ===
plt.figure(figsize=(8, 6))
venn2([set1, set2], set_labels=('IA', 'KB'))
plt.title('Overlap of Unique URLs Between Datasets')
venn_path = os.path.join(output_dir, 'url_overlap_venn_diagram.png')
plt.savefig(venn_path)
plt.show()

print(f"📊 Venn diagram saved to: {venn_path}")


In [ ]:
# Comparing urls by year

import pandas as pd
import re
import os
import matplotlib.pyplot as plt

# === File Paths ===
file1_path = r'data/IA_Rijksmuseum_unique_urls_per_year.csv'
file2_path = r'data/KB_URLs_200statuscode_full_cleaned.csv'
output_dir = r'data/Results/'

# === Load Data ===
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# === Normalize column names ===
if 'original' in df1.columns:
    df1['url'] = df1['original']
if 'original' in df2.columns:
    df2['url'] = df2['original']

# === Extract Year ===
def extract_year_from_timestamp(ts):
    match = re.match(r'(\d{4})', str(ts))
    return match.group(1) if match else 'Unknown'

if 'timestamp' in df1.columns:
    df1['year'] = df1['timestamp'].apply(extract_year_from_timestamp)
if 'crawl_year' in df2.columns:
    df2['year'] = df2['crawl_year'].astype(str)

# === Count URLs Per Year ===
urls_per_year_1 = df1['year'].value_counts().rename('IA').sort_index()
urls_per_year_2 = df2['year'].value_counts().rename('KB').sort_index()

# === Combine into one DataFrame ===
combined = pd.concat([urls_per_year_1, urls_per_year_2], axis=1).fillna(0).astype(int)
combined.index.name = 'Year'

# === Save as CSV ===
csv_output_path = os.path.join(output_dir, 'url_count_per_year_comparison.csv')
combined.to_csv(csv_output_path)

# === Plot Bar Chart ===
combined.plot(kind='bar', figsize=(12, 6), title='Total URLs per Year by Dataset')
plt.ylabel('Number of URLs')
plt.xlabel('Year')
plt.xticks(rotation=45)
plt.tight_layout()

# === Save plot ===
chart_path = os.path.join(output_dir, 'url_count_per_year_comparison.png')
plt.savefig(chart_path)
plt.show()

# === Final Message ===
print("✅ Yearly URL comparison completed.")
print(f"📄 CSV saved to: {csv_output_path}")
print(f"📊 Chart saved to: {chart_path}")


In [ ]:
# same 2001-2018

import pandas as pd
import re
import os
import matplotlib.pyplot as plt

# === File Paths ===
file1_path = r'/home/research/Desktop/notebooks/KB_Exported/Comparison_KB_IA/IA_Rijksmuseum_unique_urls_per_year.csv'
file2_path = r'/home/research/Desktop/notebooks/KB_Exported/Comparison_KB_IA/KB_URLs_200statuscode_full_cleaned.csv'
output_dir = r'/home/research/Desktop/notebooks/KB_Exported/Comparison_KB_IA/Results/'

# === Load Data ===
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# === Normalize column names ===
if 'original' in df1.columns:
    df1['url'] = df1['original']
if 'original' in df2.columns:
    df2['url'] = df2['original']

# === Extract Year ===
def extract_year_from_timestamp(ts):
    match = re.match(r'(\d{4})', str(ts))
    return match.group(1) if match else 'Unknown'

if 'timestamp' in df1.columns:
    df1['year'] = df1['timestamp'].apply(extract_year_from_timestamp)
if 'crawl_year' in df2.columns:
    df2['year'] = df2['crawl_year'].astype(str)

# === Filter years 2011–2024 ===
years_filter = [str(y) for y in range(2001, 2019)]
df1_filtered = df1[df1['year'].isin(years_filter)]
df2_filtered = df2[df2['year'].isin(years_filter)]

# === Count URLs Per Year ===
urls_per_year_1 = df1_filtered['year'].value_counts().rename('IA').sort_index()
urls_per_year_2 = df2_filtered['year'].value_counts().rename('KB').sort_index()

# Ensure all years 2011–2024 are included, even if 0
index_years = pd.Index(years_filter, name='Year')
urls_per_year_1 = urls_per_year_1.reindex(index_years, fill_value=0)
urls_per_year_2 = urls_per_year_2.reindex(index_years, fill_value=0)

# === Combine into one DataFrame ===
combined = pd.concat([urls_per_year_1, urls_per_year_2], axis=1).astype(int)

# === Save as CSV ===
csv_output_path = os.path.join(output_dir, 'url_count_2001_2018_comparison.csv')
combined.to_csv(csv_output_path)

# === Plot Bar Chart ===
combined.plot(kind='bar', figsize=(12, 6), title='Total URLs per Year (2001–2018)')
plt.ylabel('Number of URLs')
plt.xlabel('Year')
plt.xticks(rotation=45)
plt.tight_layout()

# === Save plot ===
chart_path = os.path.join(output_dir, 'url_count_2001_2018_comparison.png')
plt.savefig(chart_path)
plt.show()

# === Final Message ===
print("✅ Yearly URL comparison (2001–2018) completed.")
print(f"📄 CSV saved to: {csv_output_path}")
print(f"📊 Chart saved to: {chart_path}")

In [ ]:
# distribution and proportion of the file formats
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- Paths to your aggregated 2-row tables (HTML / IMAGE) ---
ia_path = r'data/Rijksmuseum_file_formats_per_year_aggregated.csv"
kb_path = r'data/KB_URLs_200statuscode_full_cleaned_Formats_aggregated.csv"

# --- Load data and set index = format ---
ia = pd.read_csv(ia_path).set_index("format")
kb = pd.read_csv(kb_path).set_index("format")

# Make sure all non-format columns are numeric
ia = ia.apply(pd.to_numeric, errors="ignore")
kb = kb.apply(pd.to_numeric, errors="ignore")

years_full = list(range(1999, 2026))  # 1999–2025

def prep_props(df):
    """Return two lists (html_props, image_props) aligned with years_full."""
    html_props = []
    image_props = []
    for y in years_full:
        col = str(y)
        if col in df.columns:
            html_val = df.loc["HTML", col] if "HTML" in df.index else 0
            image_val = df.loc["IMAGE", col] if "IMAGE" in df.index else 0
            total = html_val + image_val
            if total > 0:
                html_props.append(html_val / total)
                image_props.append(image_val / total)
            else:
                html_props.append(np.nan)   # no bar
                image_props.append(np.nan)
        else:
            html_props.append(np.nan)       # no data for this year
            image_props.append(np.nan)
    return np.array(html_props), np.array(image_props)

ia_html, ia_image = prep_props(ia)
kb_html, kb_image = prep_props(kb)

x = np.arange(len(years_full))
width = 0.8  # single wide bar per archive (since IA and KB are on separate plots)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- IA plot ---
ax = axes[0]
ax.bar(x, ia_html, width, label="HTML", color="#1f77b4")
ax.bar(x, ia_image, width, bottom=ia_html, label="IMAGE", color="#ff7f0e")
ax.set_ylabel("Proportion")
ax.set_title("Proportion of HTML and IMAGE per Year – Internet Archive (IA)")
ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", linewidth=0.5)
ax.legend()

# --- KB plot ---
ax = axes[1]
ax.bar(x, kb_html, width, label="HTML", color="#1f77b4")
ax.bar(x, kb_image, width, bottom=kb_html, label="IMAGE", color="#ff7f0e")
ax.set_ylabel("Proportion")
ax.set_title("Proportion of HTML and IMAGE per Year – KB")
ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", linewidth=0.5)
ax.legend()

# Shared x-axis
plt.xticks(x, years_full, rotation=45)
axes[1].set_xlabel("Year")

plt.tight_layout()
plt.show()


In [ ]:
# wordcloud of segments
# ================================
# Word cloud from IA URL segments
# ================================

# If you don't have wordcloud installed yet, run this ONCE in a separate cell:
# !pip install wordcloud

import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Load data ---

csv_path = r'data/IA_url_segment_counts_by_year_HTML_only_top15_segment1_to_4.csv"

df = pd.read_csv(csv_path)

# Check columns if needed:
# print(df.columns)

# --- 2. Prepare frequency dictionary ---

segment_cols = ['Segment 1', 'Segment 2', 'Segment 3', 'Segment 4']
freq_col = 'total'   # column with frequency / count

word_freq = {}

for _, row in df.iterrows():
    total = row[freq_col]
    # Skip rows with NaN / missing total
    if pd.isna(total):
        continue

    for col in segment_cols:
        if col in df.columns:
            word = row[col]
            # Skip empty / NaN segments
            if pd.isna(word):
                continue
            word = str(word).strip()
            if not word:
                continue

            # Add this row's total to the word frequency
            word_freq[word] = word_freq.get(word, 0) + total

# --- 3. Generate word cloud ---

wc = WordCloud(
    width=1200,
    height=600,
    background_color="white"  # you can change to "black" or others if you like
)

wc = wc.generate_from_frequencies(word_freq)

# --- 4. Visualise ---

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word cloud of IA URL segments (HTML only, weighted by 'total')", fontsize=16)
plt.tight_layout()
plt.show()
